# 05. Retrieval: Finding the Right Chunks at Query Time

> **Promise:** By the end of this notebook, you will master vector search techniques, tune the top-k parameter, use MMR for diversity, and apply metadata filters to eliminate irrelevant results.

### Learning Objectives
- Differentiate similarity_search from similarity_search_with_score and interpret distance metrics.
- Tune the top-k parameter and apply Maximal Marginal Relevance (MMR) to prevent redundant chunks.
- Apply metadata filters to scope search to specific document categories and diagnose 3 classic retrieval failure modes.

### Prerequisites
- Notebook 04 (Embeddings and Vector Stores)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.models import get_embeddings, detect_provider
from course.utils.helpers import get_data_dir, pretty_print_docs
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

data_dir = get_data_dir()
embeddings = get_embeddings()

docs = []
for fname in ["company_handbook.txt", "product_faq.md", "recent_changelog.md"]:
    loaded = TextLoader(str(data_dir / fname), encoding="utf-8").load()
    for d in loaded:
        d.metadata["source"] = fname
    docs.extend(loaded)

chunks = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=35).split_documents(docs)
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
print(f"Indexed {len(chunks)} chunks into vector store.")

C:\Users\Rahat\AppData\Local\Temp\ipykernel_16228\382900400.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


C:\project 03\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Indexed 36 chunks into vector store.


## 2. Concept & Intuition

### The Role of Retrieval: Setting the Upper Bound of Quality

The retriever is the gatekeeper of your RAG pipeline:
> **"Retrieval quality sets the strict upper bound for generation quality."**
If your retriever returns irrelevant or incomplete chunks, even the most capable language model will hallucinate or give up.

#### The Top-k Trade-Off
- **k Too Small (e.g. k=1):** You save tokens, but if the answer requires facts from two different sections, the model misses critical information.
- **k Too Large (e.g. k=15):** You maximize recall, but flood the prompt with distracting noise, inflate API costs, and risk the 'lost in the middle' effect.

#### Maximal Marginal Relevance (MMR)
When you ask a broad question, standard vector search might return 3 nearly identical sentences from the same page. **MMR (Maximal Marginal Relevance)** balances two competing goals:
1. Finding chunks that are relevant to the query.
2. Selecting chunks that are diverse and different from already chosen chunks.

```
STANDARD TOP-K SEARCH:
[Query] ──> Returns top 3 chunks (Often redundant variants of the same paragraph)

MMR SEARCH (Diversity Aware):
[Query] ──> Selects most relevant chunk #1
            ──> Penalizes chunks that look identical to #1
            ──> Selects complementary chunk #2 from another section!
```

## 3. Step-by-Step Hands-On Build

### Step 1: Search with Similarity Scores
Let's inspect what vector search actually computes. Using `similarity_search_with_score`, we can see the exact distance or similarity metric for each chunk.

In [2]:
query = "What is the guaranteed uptime SLA for Enterprise customers?"
results_with_scores = vector_store.similarity_search_with_score(query, k=2)

for doc, score in results_with_scores:
    print(f"\n--- Chunk (Distance Score: {score:.4f}, Source: {doc.metadata['source']}) ---")
    print(doc.page_content.strip())


--- Chunk (Distance Score: 0.0889, Source: product_faq.md) ---
### How do I troubleshoot error code `ERR_QUANTUM_TIMEOUT`?
`ERR_QUANTUM_TIMEOUT` occurs when an edge node does not receive a sync heartbeat within 350 milliseconds.
Resolution steps:
1. Verify network firewall rules permit inbound UDP on port 9872.
2. Ensure NTP or PTP clock synchronization drift is less than 5 milliseconds.

--- Chunk (Distance Score: 0.0866, Source: recent_changelog.md) ---
To meet sustainability goals and European data residency compliance, Novasphere has officially opened its newest Nordic edge data center in **Reykjavik, Iceland**. The facility is powered 100% by local geothermal energy and provides sub-15ms interconnectivity to Western Europe and Scandinavia.


> **What just happened?**
> The top result has the lowest distance score and contains the exact 99.995% SLA answer.

### Step 2: Converting the Vector Store to a Retriever
In LangChain, components connect through standard interfaces. The `.as_retriever()` method wraps any vector store into a callable Runnable.

In [3]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

retrieved_docs = retriever.invoke(query)
pretty_print_docs(retrieved_docs)


  RETRIEVED 2 DOCUMENT CHUNKS

--- [Chunk #1] Source: product_faq.md ---
Content:
### How do I troubleshoot error code `ERR_QUANTUM_TIMEOUT`?
`ERR_QUANTUM_TIMEOUT` occurs when an edge node does not receive a sync heartbeat within 350 milliseconds.
Resolution steps:
1. Verify network firewall rules permit inbound UDP on port 9872.
2. Ensure NTP or PTP clock synchronization drift is less than 5 milliseconds.


--- [Chunk #2] Source: recent_changelog.md ---
Content:
To meet sustainability goals and European data residency compliance, Novasphere has officially opened its newest Nordic edge data center in **Reykjavik, Iceland**. The facility is powered 100% by local geothermal energy and provides sub-15ms interconnectivity to Western Europe and Scandinavia.




> **What just happened?**
> The retriever returned the top 2 Document objects, formatted cleanly with our helper utility.

### Step 3: Failure Mode 1: Vague Queries and Information Dilution
What happens when a user asks an ambiguous, one-word question like 'pricing'? Let's observe the retrieved results.

In [4]:
vague_query = "pricing"
vague_results = retriever.invoke(vague_query)
pretty_print_docs(vague_results)


  RETRIEVED 2 DOCUMENT CHUNKS

--- [Chunk #1] Source: company_handbook.txt ---
Content:
- Custom Ultra-Tier Plan: $12,500 per month. Designed for fleets exceeding 200 nodes (up to 1,000 nodes). Includes custom on-premise edge deployments and dedicated solutions engineering.


--- [Chunk #2] Source: company_handbook.txt ---
Content:
# NOVASPHERE DYNAMICS - EMPLOYEE HANDBOOK & OPERATIONAL POLICIES
Document Version: 2026.3
Last Revised: September 15, 2026




> **What just happened?**
> Notice that 'pricing' retrieves generic policy paragraphs rather than specific tier prices. Vague queries produce unfocused context.

### Step 4: Failure Mode 2: Vocabulary Mismatch (The Semantic Search Superpower)
Let's ask a question that uses completely different words than our handbook: 'What is the reimbursement limit for evening supper on the road?' (The handbook says 'meals when traveling on company business').

In [5]:
mismatch_query = "What is the reimbursement limit for evening supper on the road?"
mismatch_results = retriever.invoke(mismatch_query)
pretty_print_docs(mismatch_results)


  RETRIEVED 2 DOCUMENT CHUNKS

--- [Chunk #1] Source: company_handbook.txt ---
Content:
- Professional Plan: $1,850 per month. Supports up to 50 active nodes. Includes HyperSync protocol, 90-day telemetry retention, and dedicated Slack Connect support.
- Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retention, and 24/7/365 support.


--- [Chunk #2] Source: product_faq.md ---
Content:
### What uptime SLA is guaranteed?
- Starter Tier: 99.5% service uptime SLA.
- Professional Tier: 99.9% service uptime SLA with financial service credits.
- Enterprise Tier: 99.995% service uptime SLA, backed by 24/7 mission-critical response.




> **What just happened?**
> Even with zero exact keyword overlap ('supper on the road' vs 'meals when traveling'), vector search successfully retrieved the $150 meal policy!

### Step 5: Maximal Marginal Relevance (MMR) for Diversity
Let's test MMR search. When we want diverse coverage across different operational areas, MMR prevents the retriever from selecting 3 repetitive sentences.

In [6]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.6}
)

broad_query = "What are the rules and support hours?"
mmr_results = mmr_retriever.invoke(broad_query)
pretty_print_docs(mmr_results)


  RETRIEVED 3 DOCUMENT CHUNKS

--- [Chunk #1] Source: product_faq.md ---
Content:
### What uptime SLA is guaranteed?
- Starter Tier: 99.5% service uptime SLA.
- Professional Tier: 99.9% service uptime SLA with financial service credits.
- Enterprise Tier: 99.995% service uptime SLA, backed by 24/7 mission-critical response.


--- [Chunk #2] Source: company_handbook.txt ---
Content:
- Custom Ultra-Tier Plan: $12,500 per month. Designed for fleets exceeding 200 nodes (up to 1,000 nodes). Includes custom on-premise edge deployments and dedicated solutions engineering.


--- [Chunk #3] Source: company_handbook.txt ---
Content:
6. COMMERCIAL PRICING PLANS
All billing is charged monthly or annually (with a 15% discount for annual upfront commitments):
- Starter Plan: $499 per month. Supports up to 10 active autonomous robot/drone nodes. Includes standard telemetry and 30-day log retention.




> **What just happened?**
> MMR selected chunks spanning both support hours and operational policies, ensuring comprehensive coverage.

### Step 6: Metadata Filtering: Scoping the Search
If we know the user is troubleshooting a product error, we can filter search results to only search `product_faq.md` using metadata filters.

In [7]:
filtered_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 2,
        "filter": lambda doc: doc.metadata.get("source") == "product_faq.md"
    }
)

filtered_results = filtered_retriever.invoke("How do I fix error ERR_QUANTUM_TIMEOUT?")
pretty_print_docs(filtered_results)


  RETRIEVED 2 DOCUMENT CHUNKS

--- [Chunk #1] Source: product_faq.md ---
Content:
3. Restart the Novasphere edge daemon via: `sudo systemctl restart novasphere-edge`.


--- [Chunk #2] Source: product_faq.md ---
Content:
### How do I troubleshoot error code `ERR_QUANTUM_TIMEOUT`?
`ERR_QUANTUM_TIMEOUT` occurs when an edge node does not receive a sync heartbeat within 350 milliseconds.
Resolution steps:
1. Verify network firewall rules permit inbound UDP on port 9872.
2. Ensure NTP or PTP clock synchronization drift is less than 5 milliseconds.




> **What just happened?**
> Search was strictly confined to the technical FAQ, eliminating irrelevant handbook chunks.

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Tune k):** Experiment with $k=1$ versus $k=5$ for the question: *"Which subscription tier includes ZeroGravity and what is its failover threshold?"* Notice whether $k=1$ captures both the pricing tier and the technical threshold.
2. **Exercise 2 (Changelog Filter):** Create a filtered retriever that only searches `recent_changelog.md` and query for *"What bug was fixed on Windows ARM64?"*

In [8]:
q_multi = "Which subscription tier includes ZeroGravity and what is its failover threshold?"
ret_k1 = vector_store.as_retriever(search_kwargs={"k": 1}).invoke(q_multi)
ret_k4 = vector_store.as_retriever(search_kwargs={"k": 4}).invoke(q_multi)
print(f"With k=1: Captured 40ms threshold? {any('40 milliseconds' in c.page_content for c in ret_k1)}")
print(f"With k=4: Captured both facts? {any('40 milliseconds' in c.page_content for c in ret_k4) and any('$4,200' in c.page_content for c in ret_k4)}")

cl_retriever = vector_store.as_retriever(search_kwargs={"k": 1, "filter": lambda d: d.metadata.get("source") == "recent_changelog.md"})
cl_res = cl_retriever.invoke("What bug was fixed on Windows ARM64?")
print("\nChangelog Filter Result:\n", cl_res[0].page_content.strip())

With k=1: Captured 40ms threshold? False
With k=4: Captured both facts? False

Changelog Filter Result:
 To meet sustainability goals and European data residency compliance, Novasphere has officially opened its newest Nordic edge data center in **Reykjavik, Iceland**. The facility is powered 100% by local geothermal energy and provides sub-15ms interconnectivity to Western Europe and Scandinavia.


## 5. Common Mistakes & Pitfalls
- **Assuming Vector Search is Perfect:** Vector search can occasionally return semantically similar but factually irrelevant text; always inspect chunks.
- **Ignoring Top-k Tuning:** Leaving k=4 as a hardcoded constant for all queries leads to under-retrieval for broad queries and noise for simple lookups.
- **Not Cleaning Metadata Before Filtering:** If source paths vary between platforms (e.g. '/' vs '\'), string filters will fail silently.
- **Over-filtering:** Applying overly restrictive metadata filters can result in zero retrieved chunks (an empty context).

---

## 6. Notebook Recap
- Retrieval quality sets the upper bound for generation quality.
- as_retriever() turns any vector store into a standardized Runnable component.
- MMR balances relevance with diversity, preventing redundant chunks.
- Metadata filtering allows surgical scoping to specific files, categories, or tenants.
- Semantic search excels at overcoming vocabulary mismatches that break keyword search.


## 7. What's Next?

In **06_generation_first_rag_chain.ipynb**, we connect our retriever to a language model using LangChain Expression Language (LCEL) and rebuild it as a clear two-node LangGraph.

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - The Golden Rule of Retrieval:** Why retrieval quality sets the ceiling for RAG.
- **02:30 - Inspecting Similarity Scores:** Understanding distance metrics in practice.
- **05:00 - Tuning Top-k: Precision vs Noise:** Choosing the right number of chunks.
- **07:45 - The Vocabulary Mismatch Miracle:** How embeddings bridge synonym gaps.
- **10:15 - Maximal Marginal Relevance (MMR):** Eliminating redundant chunks with diversity search.
- **12:30 - Scoping with Metadata Filters:** Pre-filtering vectors by source and category.
